# Solution D2 · Residual diagnostics: four mysteries

**Type**: Interpret

## Objective

Implement three diagnostic tests on the residuals (Durbin-Watson, curvature, Breusch-Pagan) and use them to
diagnose four datasets: one respects the assumptions, the other three violate one each.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from scipy import stats
from common.plots import residual_panel

mysteries = ds.residual_mysteries()
{name: d.shape for name, d in mysteries.items()}

## Commented solution

In [ ]:
def residuals_of(df):
    x, y = df["x"].to_numpy(), df["y"].to_numpy()
    A = np.column_stack([np.ones(len(x)), x])
    beta = np.linalg.lstsq(A, y, rcond=None)[0]
    return x, y - A @ beta


def _regression_r2(B, z):
    beta = np.linalg.lstsq(B, z, rcond=None)[0]
    return 1 - ((z - B @ beta) ** 2).sum() / ((z - z.mean()) ** 2).sum()


def durbin_watson(e):
    return np.sum(np.diff(e) ** 2) / np.sum(e ** 2)


def curvature_r2(x, e):
    return _regression_r2(np.column_stack([np.ones_like(x), x, x ** 2]), e)


def breusch_pagan(x, e):
    r2 = _regression_r2(np.column_stack([np.ones_like(x), x]), e ** 2)
    stat = len(x) * r2
    return stat, stats.chi2.sf(stat, df=1)

In [ ]:
rows = {}
for name, d in mysteries.items():
    x, e = residuals_of(d)
    rows[name] = {"dw": durbin_watson(e), "curv_r2": curvature_r2(x, e), "p_bp": breusch_pagan(x, e)[1]}
table = pd.DataFrame(rows).T
pd.options.display.float_format = "{:.4g}".format
table

Every row has exactly **one** statistic out of range: A has a tiny Breusch-Pagan p-value (the variance depends on
x), C has a curvature R² close to 1 (the residuals are shaped like a parabola), D has a Durbin-Watson close to 0 (consecutive
residuals look alike). B triggers nothing.

In [ ]:
def classify(row):
    if row.dw < 1.2:
        return "autocorrelation"
    if row.curv_r2 > 0.10:
        return "nonlinearity"
    if row.p_bp < 0.01:
        return "heteroscedasticity"
    return "ok"

diagnosis = {name: classify(row) for name, row in table.iterrows()}
diagnosis

In [ ]:
for name in ["A", "C", "D"]:
    x, e = residuals_of(mysteries[name])
    d = mysteries[name]
    residual_panel(d["y"], d["y"] - e, f"Dataset {name}: {diagnosis[name]}")

## Check

In [ ]:
with verify():
    check(close(durbin_watson(np.array([1.0, -1.0, 1.0, -1.0])), 3.0), "durbin_watson is correct on a known case",
          "durbin_watson([1,-1,1,-1]) must equal 12/4 = 3")
    rng_t = np.random.default_rng(0)
    x_t = rng_t.uniform(-3, 3, 300)
    check(curvature_r2(x_t, x_t ** 2 - (x_t ** 2).mean()) > 0.99, "curvature_r2 recognizes a perfect parabola",
          "regressing on [1, x, x^2] an exact parabola must give R2 = 1")
    x_h = rng_t.uniform(1, 10, 500)
    check(breusch_pagan(x_h, rng_t.normal(size=500) * x_h)[1] < 1e-3, "breusch_pagan recognizes growing variance",
          "with errors whose standard deviation grows with x the p-value must be tiny")
    check(breusch_pagan(x_h, rng_t.normal(size=500))[1] > 0.01, "breusch_pagan does not fire on homoscedastic errors",
          "with errors independent of x the p-value should not be small")
    check(list(table.index) == ["A", "B", "C", "D"] and list(table.columns) == ["dw", "curv_r2", "p_bp"],
          "the table has the required index and columns", "index A-D and columns dw, curv_r2, p_bp")
    check(diagnosis == {"A": "heteroscedasticity", "B": "ok", "C": "nonlinearity", "D": "autocorrelation"},
          "correct diagnosis for all four datasets",
          "look at which statistic is out of range in each row of `table`: at most one should fire")
done()

## What you learned

- **Nonlinearity**: the residuals against x have a shape (here a parabola). Remedies: add polynomial terms or
  transform the features. The predictions are systematically wrong: it is the most serious case.
- **Heteroscedasticity**: the spread of the residuals changes with x (funnel). The estimates remain unbiased, but the classical
  standard errors are wrong: you need robust errors, weighted least squares or a transformation of the target.
- **Autocorrelation**: consecutive errors are not independent (typical of time data). Here too the coefficient
  estimates are reasonable but the uncertainty is underestimated, and the residuals have a "wave" pattern.
- The three statistics are almost orthogonal: the right one fires, the others stay in range. In real data they often
  fire together and everything gets worse.

## Common pitfall

Fixating on the Q-Q plot and on the normality of the residuals: in dataset A the normality test (Shapiro) rejects, in B it does not, but
the problem of A is not the shape of the distribution, it is the non-constant variance. Normality is the least important
of the assumptions for predicting. A test that fires does not tell you **which** assumption is violated: you need the right statistic.

## Interview question

<details><summary>You have a regression on daily sales data and the plot of the residuals against time shows regular waves. Which assumption is violated and what are the consequences?</summary>

The independence of the errors: there is autocorrelation. The coefficient estimates remain reasonable but the standard errors
are underestimated, so p-values and intervals are too optimistic. I would confirm with Durbin-Watson, and move to a
model that handles the temporal dependence (AR errors, lagged features, seasonal components). For validation I would
use TimeSeriesSplit, not a random K-fold.

</details>